[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/05_coral.ipynb)

In [ ]:
# Install CMGDB and the paper repository (the clone carries the saved model
# weights the notebooks recompute from). Running locally inside the repo,
# skip this cell.
!git clone -q --depth 1 --branch paper https://github.com/begelb/latent_dynamics.git
!pip install -q CMGDB
!pip install -q -e latent_dynamics
%cd latent_dynamics

# An editable install only becomes importable after a kernel restart, so
# import the package straight from the clone instead.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

# Section 5.4 - Red Coral population model

## What this notebook shows

A demographic model of **Mediterranean red coral** (paper section 5.4) built
from real field data, with **thirteen age classes** (a 13-dimensional system).
We learn a **one-dimensional** latent model -- the bistability lives on a single
latent coordinate -- and compute its Morse graph, which has two minimal nodes
(two stable equilibria).

The default loads the model trained with **adaptive sampling**: the base design
$\mathcal{D}(500)$ augmented with 300 adaptively-chosen samples (section 5.4.2).
The histograms at the end compare the base design against the augmented one.

### How to run

Set `MODE` in the parameters cell, then Run All.

| `MODE` | what it does | typical cost |
|--------|--------------|--------------|
| `"quick"` | recompute the Morse graph of the *saved* model on the coarse `QUICK_SUBDIV` grid | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` (paper value by default) | minutes |
| `"retrain"` | train a fresh model, then compute its Morse graph at your `SUBDIV` | minutes-hours (GPU recommended) |

**Coarse grids can merge nearby recurrent sets and change the Morse graph**,
so `quick` is a preview, not a paper-quality result. Nothing a notebook does
ever touches the preserved paper trees: recomputes land under
`output/notebooks/<experiment>/`.

> **Retrain cost:** a full coral cell trains and computes Morse on the order of ~50-87 minutes.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "quick"             # "quick" | "morse" | "retrain"
SEED = 16                  # the published coral run; other seeds ship only legacy checkpoints
QUICK_SUBDIV = (6, 8, 10)  # MODE="quick": coarse preview grid, runs in seconds
SUBDIV = (8, 8, 12)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # the paper value
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
TRAIN_FILE = "train_500"   # which sampling design to load or train
# ===========================================================================

REPLAY_CONFIG  = "coral_basic"
RETRAIN_CONFIG = "coral_basic"

## The system

A stage-structured demographic model of a red coral population: thirteen size
classes, whose transitions are fixed by the published demography rather than by
free parameters. The latent model is **one**-dimensional here, so the Morse
sets are intervals and the figures are bands rather than boxes.

In [ ]:
# ---- the system -----------------------------------------------------------
# The coral demography carries no free parameters: the transition rates are
# fixed by the published model, so only the latent dimension is a choice here.
LATENT_DIMS = 1

from latentdynamics.config import load_config
from latentdynamics.systems import build_system

SYSTEM_PARAMS = {}
system = build_system("coral", SYSTEM_PARAMS)
print(f"ambient dimension {system.dim} (size classes), latent dimension {LATENT_DIMS}")

## The autoencoder and its latent map

An encoder, a decoder, and a latent map trained together so the latent map is
an $\epsilon$-approximate semiconjugacy to the full system on the data.

In [ ]:
# ---- the autoencoder (paper values) --------------------------------------
HIDDEN_SHAPES = [64, 64, 64]        # per component: encoder, latent map, decoder
LOSS_WEIGHTS = [10.0, 10.0, 1.0]    # (w1, w2, w3): reconstruction, latent step, cycle
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EPOCHS = 1000
PATIENCE = 100

print(f"{system.dim} -> {LATENT_DIMS} -> {system.dim}, hidden {HIDDEN_SHAPES} per component")
print(f"loss weights {LOSS_WEIGHTS}, Adam lr {LEARNING_RATE}, batch {BATCH_SIZE}")

## The data

Pairs $(x, f(x))$ along trajectories from sampled initial conditions: the model
only ever sees one-step transitions, never the map itself.

In [ ]:
# ---- the data (paper values) ---------------------------------------------
# Coral is a data-scaling study: TRAIN_FILE in the parameters cell picks which
# sampling design to load, and how much data it holds is the experiment.
N_TRAIN = 500         # initial conditions in the training design
N_VAL = 10000         # validation initial conditions
N_ITERATIONS = 20     # steps per trajectory

print(f"design {TRAIN_FILE}: {N_TRAIN} train / {N_VAL} validation, {N_ITERATIONS} steps each")

# Everything above is fed to the pipeline as config overrides, so `retrain`
# below trains exactly the model described here.
OVERRIDES = {
    "arch": {
        "low_dims": LATENT_DIMS,
        "encoder": {"hidden_shapes": HIDDEN_SHAPES},
        "latent_map": {"hidden_shapes": HIDDEN_SHAPES},
        "decoder": {"hidden_shapes": HIDDEN_SHAPES},
    },
    "training": {
        "loss_weights": LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
    },
    "data": {
        "n_samples_val": N_VAL,
        "n_iterations": N_ITERATIONS,
        
    },
}

# Flag anything that no longer matches the paper's configuration.
paper = load_config(RETRAIN_CONFIG)
drift = []

if LATENT_DIMS != paper.arch.low_dims:
    drift.append(f"latent dims {paper.arch.low_dims}")
if HIDDEN_SHAPES != paper.arch.encoder.hidden_shapes:
    drift.append(f"encoder hidden {paper.arch.encoder.hidden_shapes}")
for name, value, reference in [
    ("loss weights", LOSS_WEIGHTS, paper.training.loss_weights),
    ("learning rate", LEARNING_RATE, paper.training.learning_rate),
    ("batch size", BATCH_SIZE, paper.training.batch_size),
    ("epochs", EPOCHS, paper.training.epochs),
    ("patience", PATIENCE, paper.training.patience),
    ("validation trajectories", N_VAL, paper.data.n_samples_val),
    ("iterations", N_ITERATIONS, paper.data.n_iterations),
]:
    if value != reference:
        drift.append(f"{name} {reference}")
print("\nmatches the paper's configuration" if not drift
      else "\ndiffers from the paper, which uses: " + "; ".join(drift))

## Training

`quick` and `morse` load the paper's trained weights. `retrain` runs the data,
scaling, training and diagnostic stages and then CMGDB itself, at the
`SUBDIV` set above.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

if MODE in ("quick", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED, train_file=TRAIN_FILE)
elif MODE == "retrain":
    # The retrain pipeline runs CMGDB itself, at the SUBDIV set above.
    OVERRIDES["cmgdb"] = {
        "subdiv_init": SUBDIV[0],
        "subdiv_min": SUBDIV[1],
        "subdiv_max": SUBDIV[2],
    }
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        train_file=TRAIN_FILE,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose", "morse"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Training curves

Total loss and its terms, per epoch. In `quick` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph

CMGDB subdivides the latent rectangle into boxes and builds the directed graph
on them induced by the latent map: box `B` points at every box meeting an
enclosure of `g(B)`. The **Morse sets** are that graph's strongly connected
components, and the **Morse graph** is its condensation, with a Conley index on
each node. `ComputeConleyMorseGraph` returns both -- the map graph is not an
extra step, it *is* the computation.

The grid comes from `QUICK_SUBDIV` in `quick` mode and from `SUBDIV`
otherwise, chosen here rather than inherited from the training config.

The box map evaluates the network on the corner lattice in batches -- one call
per batch instead of one per box. CMGDB (>= 1.5.0) caches the transition graph in one block with
automatic edge reservation, so no environment tuning is needed; deep grids
are limited by memory alone.

In [ ]:
# quick and morse recompute the Morse graph of the saved model; retrain already
# computed it during the pipeline run above. Either way the artifacts live in
# the notebook playground, never in the preserved paper trees.
if MODE == "retrain":
    run = exp
else:
    subdiv = QUICK_SUBDIV if MODE == "quick" else SUBDIV
    run = exp.recompute_morse(subdiv=subdiv)
MG_DIR = run.morse_dir
print(f"artifacts -> {MG_DIR}")

### What came out

Each Morse set with its Conley index, how many boxes it occupies, and what it
flows into. A node with no outgoing edges is minimal: an attractor.

In [ ]:
import numpy as np

from latentdynamics.analysis import MorseGraph

graph = MorseGraph.from_dot(MG_DIR / "morse_graph")
boxes = np.atleast_2d(np.loadtxt(MG_DIR / "morse_sets", delimiter=","))
counts = dict(zip(*np.unique(boxes[:, -1].astype(int), return_counts=True)))

print(f"{len(graph.nodes)} Morse sets, {len(graph.minimal)} minimal")
for node in graph.nodes:
    index = graph.labels.get(node, "?").split(":", 1)[-1].strip()
    edges = sorted(graph.edges.get(node, []))
    flow = "minimal" if not edges else "-> " + ", ".join(str(e) for e in edges)
    print(f"  {node}: {index:<16} {counts.get(node, 0):>8d} boxes   {flow}")

## Figures

Rendered from the DOT and CSV above with the paper's palette and axis labels,
so a recomputed run and the paper figure come out of the same code. `BOX_SCALE`
only affects drawing: it inflates Morse sets too small to see.

In [ ]:
from latentdynamics.replay import show_image

figs = run.render_morse(box_scale=BOX_SCALE)
show_image(figs.morse_graph_png, width=600)
for png in (p for p in figs.morse_sets_paths if p.suffix == ".png"):
    show_image(png, width=720)

## Run provenance

`metrics` reports the section 5.4.1 success check: `a0`/`a1` should be `true`
(each reference attractor sits in a unique minimal Morse set) and `r` should be
`false`.

In [ ]:
run.diagnostics()